Un **cifraro a blocchi** rappresenta una generalizzazione dei cifrari a sostituzione. 

Nel cifrario a sostituzione classico, ogni lettera è associata a un'altra. In quello a blocchi si lavora invece su **sequenze di bit di lunghezza fissa**: ad ogni blocco di bit viene associato un altro blocco di bit della stessa lunghezza.

Sia $n$ la dimensione del blocco. La cifratura è $C = E(K, M)$ dove:
- $M$ è un blocco di plaintext di $n$ bit,
- $K$ è la chiave segreta (per i nostri scopi assumiamo ancora **crittografia simmetrica**, i.e. sia mittente che destinatario condividono la stessa chiave),
- $C$ è il blocco di ciphertext, sempre di $n$ bit, risultante dalla cifratura.

Allora la decifratura deve saper recuperare $M = D(K, C)$.  
La proprietà fondamentale è quindi $D(K, E(K, M)) = M$.

Il collegamento con la sostituzione è che **la chiave determina quale associazione applicare tra i blocchi in chiaro e quelli cifrati**: l'algoritmo è pubblico ma la particolare trasformazione dipende dalla chiave segreta.

es. (piccolo) $E(K, 001) = 010$. Allora il destinatario, conoscendo $K$, deve poter applicare la trasformazione inversa $D(K,010) = 001$.

Il cifrario a blocchi, considerato come componente di base, trasforma **un singolo blocco**. Se il messaggio è più lungo allora bisogna stabilire come usare il componente sui diversi blocchi: si parla dei **modi di funzionamento** (li vediamo dopo). Per ora focalizziamoci sul singolo blocco.

### Perché la trasformazione deve essere una permutazione
Sia $S = {0,1}^n$ l'insieme di tutti i possibili blocchi di $n$ bit. Allora il numero di blocchi possibili è $|S| = 2^n$ (ogni bit o 0 o 1).  
Per una chiave fissata $K$, il cifrario realizza una funzione
$P_K: S \to S$ che associa ad ogni blocco di plaintext un blocco di ciphertext.

Ma è **necessario che la funzione $P_K$ sia biettiva**: supponiamo infatti che invece due plaintext diversi producano lo stesso ciphertext:
$$E(K, 001) = E(K, 110) = 100$$
Ma allora il destinatario che riceve $100$ non sa quale messaggio recuperare, perché non può determinarlo in modo univoco (ambiguità).  
Quindi in generale per una chiave fissata $K$:
$$M_1 \neq M_2 \implies E(K, M_1) \neq E(K, M_2)$$
Una trasformazione biettiva di un insieme a sé stesso è una **permutazione**, dal momento che riassocia gli elementi dello stesso insieme senza duplicarne o perderne nessuno.

<img src="imgs/perm.png" width="400">

Qui la differenza sostanziale rispetto agli hash: un hash che comprime messaggi in un digest di lunghezza fissa deve necessariamente ammettere collisioni, mentre il cifrario a blocchi deve essere invertibile.

La funzione quindi associa un blocco ad una sua possibile permutazione.  
Poiché il numero totale di blocchi è $2^n$, il numero di possibili permutazioni è $(2^n)!$ (ogni possibile permutazione rappresenta una possibile chiave, un modo di associare i blocchi in chiaro a quelli cifrati).

(le permutazioni non sono riferite alla lunghezza del blocco (es. per tre bit 000, 001, 010 etc..) ma al numero totale di blocchi possibili!  
Sempre nell'esempio dei tre bit, al primo possiamo assegnare uno qualsiasi degli 8 output, al secondo ne rimangono 7 etc.. per questo $(2^3)! = 8 \cdot 7 \cdot 6 \cdot 5 \cdot 4 \cdot 3 \cdot 2 \cdot 1 = 40320$).

Quindi con soli 3 bit esistono già 40320 permutazioni diverse, con blocchi di 8 bit il numero cresce enormemente a $256! \approx 8.58 \cdot 10^{506}$  
Di nuovo collegamento con la sostituzione delle lettere: con 26 lettere esistono $26!$ associazioni biettive (tutti i possibili modi di mappare le lettere a sé stesse), nei cifrari a blocchi al posto del 26 abbiamo $2^n$ possibili stringhe binarie.

### AES, PRPs e practical ciphers
Esistono diverse versioni in base alla dimensione della chiave: AES-128, AES-192, AES-256 (rispettivamente 128, 192 e 256 bit PER LA CHIAVE, per il blocco sempre 128 bit).

Quindi per AES il numero possibile di blocchi è $2^{128}$ mentre il numero di permutazioni possibili è $(2^{128})!$  
Tuttavia **una chiave pratica non può selezionare una qualsiasi tra queste permutazioni, troppo costoso!** I practical ciphers lavorano solo su una "piccola" famiglia di permutazioni.

In generale una chiave di $k$ bit può assumere $2^k$ valori, e ogni valore della chiave determina una permutazione. Quindi una chiave di $k$ bit può realizzare al massimo $2^k$ permutazioni distinte. 

Prendendo anche AES-256, la chiave è lunga 256 bit quindi può realizzare al massimo $2^{256}$ permutazioni distinte, ma il numero di permutazioni possibili è $(2^{128})! \gg 2^{256}$.   
Questo è proprio il punto dei cifrari pratici: **AES lavora su una famiglia limitata di permutazioni, ciascuna calcolabile efficientemente a partire dalla chiave**.

Quindi scegliere una permutazione arbitraria tra tutte quelle possibili non garantisce necessariamente che esista una chiave AES che la realizzi.

Una **PRP** (pseudorandom permutation) rappresenta una permutazione che viene essere calcolata da un algoritmo usando una chiave, ma che deve apparire come una permutazione casuale agli occhi di un attaccante con risorse limitate.  

Infatti la proprietà richiesta da una buona PRP è che **un attaccante efficiente non sappia distinguere il cifrario PRP con chiave casuale da una permutazione casuale ideale (i.e. scelta u.a.r.)**.

Un buon practical cipher deve essere considerato PRP anche se **considerano solo una piccola parte delle permutazioni possibili** (pseudocausale riguarda il comportamento osservabile, non la copertura completa dell'insieme delle permutazioni).

Inoltre, una volta fissata la chiave, la trasformazione è **deterministica**: $E(K, M)$ produce sempre lo stesso $C$ per lo stesso $M$ e la stessa chiave $K$.

(Per selezionare arbitrariamente una delle possbili permutazioni, in una implementazione ideale, servirebbero circa $log_2((2^{128})!)$ bit di informazione, un numero astronomico per $n = 128$

Ricorda inoltre che per calcolare fattoriali grossi si può usare la formula di Stirling: $n! \approx \sqrt{2 \pi n} \left(\frac{n}{e}\right)^n$) 

#### Primo Problema: il cifrario è più lungo di un blocco
Finora abbiamo considerato il cifrario a blocchi come una funzione che, fissata la chiave, trasforma il blocco di $n$ bit in un altro di $n$ bit.  
Tuttavia un messaggio è normalmente più lungo di un singolo blocco, ad es. con AED ogni blocco ha solo 128 bit...

La soluzione banale consiste nel dividere il messaggio in blocchi e cifrarli indipendentemente con la stessa chiave:
$$M = m[0] || m[1] || ... || m[l-1] \quad c[i] = E_K(m[i])$$
tale soluzione è nota come **Electronic Codebook (ECB)**.

Il problema in ciò è che, per una chiave fissata $K$, il cifrario è per definizione **deterministico**, per cui:
$$m[i] = m[j] \implies c[i] = c[j]$$
Come si vede nell'immagine ad esempio il blocco `COME` compare due volte e viene cifrato entrambe le volte come `39B7`: un attaccante che non conosce la chiave può quindi notare che i due blocchi sono uguali e dedurre informazioni sul plaintext --> se c'è un pattern riconoscibile, questo può restare visibile anche dopo la cifratura.

<img src="imgs/ecb.png" width="400">

Questo viola **IND-CPA**: l'attaccante può scegliere due messaggi della stessa lunghezza
$$M_0 = A || A, \quad M_1 = A || B, \quad A \neq B$$
Il challenger cifra uno dei due e restituisce il ciphertext. Allora l'attaccante se vede i due blocchi uguali può dedurre che il messaggio cifrato era $M_0$, altrimenti era $M_1$ (se i due blocchi sono diversi era $M_1$ necessariamente perché $E_K$ è una permutazioone --> input distinti producono output distinti).

Quindi **ECB non è IND-CPA sicuro, anche se il cifrario sottostante è una funzione PRP sicura --> NON USARE MAI ECB!!!**


#### Secondo Problema: cifrare più volte lo stesso messaggio
Supponiamo che il messaggio occupi solo un blocco --> abbiamo il problema per cui se cifriamo nuovamente lo stesso messaggio con la stessa chiave, otteniamo sempre lo stesso ciphertext $c = E_K(m)$ --> l'attaccante può riconoscere ripetizioni tra trasmissioni diverse.

Di nuovo violazione IND-CPA: l'attaccante può sapere a priori il ciphertext relativo a un messaggio $m_0$ (nel gioco IND-CPA ha a disposizione un oracolo che cifra messaggi a sua scelta), coservare il ciphertext della sfida relativo a $m_0$ o $m_1$ e dedurre quale dei due messaggi è stato cifrato.

Per superare il problema è necessario introdurre freshness, si utilizza di nuovo una quantità che cambia ad ogni cifratura: tornano gli **Initialization Vector (IV)**.

Per inserire l'IV il prof presenta la costruzione seguente, per un singolo blocco:
$$c = E_K(m \oplus IV)$$
L'ordine delle operazioni è quindi generare l'IV u.a.r., calcolare $m \oplus IV$ e infine cifrare il risultato con la chiave $K$. L'IV quindi deve **avere la stessa lunghezza del blocco**. 

Questa è una soluzione perchè **lo XOR con valore casuale rende casuale l'ingresso del cifrario, anche quando il messaggio è molto prevedibile.** (se $IV$ u.a.r. nelle stringhe di $n$ bit, allora anche $m \oplus IV$ è uniforme).

Il ciphertext trasmesso comprende anche l'IV $CT = IV || c$: ricorda infatti che **l'IV non deve essere segreto in quanto il destinatario deve poterlo conoscere per decifrare il messaggio**.

La decifratura al solito quindi è 
$$D_K(c) = m \oplus IV \quad m = D_K(c) \oplus IV$$
(funziona perché $(x \oplus y) \oplus y = x$ e perché $D_K$ è una funzione applicabile solo da chi conosce la chiave $K$).

**Esercizio**: mostrare che la seguente costruzione è insicura
$$c = E_K(m) \oplus IV \quad CT = IV || c$$
Facile: l'IV è come detto trasmesso in chiaro --> l'attaccante può eliminarlo $c \oplus IV = E_K(m)$ e quindi ottenere il ciphertext deterministico del messaggio, che volevamo nascondere.

Una dimostrazione IND-CPA è la seguente:
1. l'attaccante chiede la cifratura di $m_0$ all'oracolo, ottiene $(IV_0, c_0)$
2. calcola $y_0 = c_0 \oplus IV_0 = E_K(m_0)$
3. propone al challenger due messaggi $m_0, m_1$ con $m_0 \neq m_1$ e riceve $(IV^*, c^*)$
4. calcola $y^* = c^* \oplus IV^* 
5. Se $y^* = y_0$ allora il messaggio cifrato era $m_0$, altrimenti era $m_1$ (la distinzione riesce sempre perché $E_K$ è una permutazione, quindi input distinti producono output distinti, i.e. il cifrario è invertibile).

##### L'IV deve essere u.a.r.!
Importante sottolineare che per la costruzione appena vista **non basta utilizzare IV diversi, devono anche essere imprevedibili**. 

Ipotizziamo ad esempio invece che l'IV sia generato in modo deterministico a partire da un contatore pubblico. Allora ipotizzando che l'attaccante conosca una precedente coppia $c = E_K(m \oplus IV)$, se può prevedere $IV'$ può scegliere
$$m' = m \oplus IV \oplus IV' \quad \text{per cui} \quad m' \oplus IV' = m \oplus IV$$
e quindi chiedendo di cifrare $m'$ all'oracolo con $IV'$ ottiene lo stesso ciphertext $c$:
$$E_K(m' \oplus IV') = E_K(m \oplus IV) = c$$
Quindi nel gioco IND-CPA l'attaccante potrà proporre due messaggi distinti $m'$ e $x$, e quando riceve il ciphertext dal challenger se era $c$ sa che il messaggio cifrato era $m'$, altrimenti era $x$ --> viola IND-CPA perché riesce a distinguere i messaggi senza conoscere la chiave.

In CBC (Cipher Block Chaining, tra poco lo vediamo) l'IV iniziale deve essere generato in modo imprevedibile rispetto alla scelta del messaggio.  
Il requisito però in generale dipende dalla **modalità**: NIST richiede IV imprevedibili per CBC e CFB, mentre per OFB richiede IV univoci sotto la stessa chiave. Quindi l'imprevedibilità in generale non è da estendere a qualsiasi uso di un cifrario a blocchi. 

Chiaramente inoltre si vuole che l'IV non si ripeta mai. In realtà matematicamente, se l'IV è u.a.r. e lungo $n$ bit, c'è probabilità di collisione (due IV uguali) dopo circa $2^{n/2}$ cifrature (paradosso del compleanno). La sicurezza considera questa probabilità (che di base è praticamente zero) imponendo comunque limiti alla quantità di dati cifrati sotto la stessa chiave.

Quindi ECB base va bene al limite se:
- messaggio più corto di un blocco (evito il problema di ripetizione dei blocchi)
- il messaggio inviato non è mai uguale a messaggi cifrati in precedenza

Chiaramente queste condizioni sono impraticabili, per questo serve l'IV e le **modalità di funzionamento** che ora vediamo.

### Risoluzione del primo problema: le **modalità di funzionamento** (modes of operation)
Una **mode of operation** stabilisce **come utilizzare un cifrario a blocchi per cifrare un messaggio intero più lungo di un blocco**.

Si distinguono quindi:
- **cifrario a blocchi** == es. AES: definisce $E_K$ e $D_K$ su un singolo blocco di $n$ bit
- **mode of operation** == es. CBC, CFB, OFB, CTR: definisce come trattare i vari blocchi, usare l'IV e costruire quindi il ciphertext complessivo

<img src="imgs/modes.png" width="500">

Most used: CBC (Cipher Block Chaining), CTR (Counter Mode).  
CFB (Cipher Feedback) and OFB (Output Feedback) interessanti e suggeriti dal NIST, ma meno usati in pratica.  
Si farà inoltre un accenno a GCM (Galois Counter Mode) e OCB (Offset Codebook Mode), che forniscono anche autenticazione oltre alla cifratura (**cifratura autenticata**).

Ricorda confidenzialità impedisce all'attaccante di apprendere il contenuto, integrità permette di rilevare se un attaccante ha modificato il messaggio, mentre autenticazione impedisce all'attaccante di impersonare il mittente.  
CBC, CTR, CFB e OFB considerati da soli forniscono solo confidenzialità, mentre la cifratura autenticata oltre anche all'autenticazione permette anche di affrontare attacchi che coinvolgono ciphertext modificati.

#### CBC (Cipher Block Chaining)
Prima di CBC una costruzione più semplice ma didattica:
$$c[i] = E_K(m[i] \oplus IV_i)$$
con un nuovo $IV$ casuale e indipendente per ogni blocco.

Allora per decifrare occorrerebbe trasmettere $(IV_0, IV_1, ...)$: si applica cioè ad ogni blocco la randomizzazione studiata nel caso singolo.  
Teoricamente funziona, ma il problema è **l'overhead**: per ogni blocco di dati cifrati trasmettiamo anche un blocco di $IV$.  
Con AES quindi per ogni blocco da 16 byte altri 16 byte: la dimensione trasmessa raddoppia.  
Inoltre generare tutti questi $IV$ richiederebbe un generatore pseudocasuale separato, richiedendo una componente crittografica aggiuntiva da progettare e usare correttamente.

Con CBC ci si chiede invece se possiamo ottenere la variazione necessaria tra blocchi sfruttando direttamente il cifrario che stiamo già usando.

CBC sta per **Cipher Block Chaining** --> i blocchi vengono concatenati tramite dipendenza crittografica, per ogni blocco si sfrutta il ciphertext del blocco precedente per randomizzare il plaintext del blocco corrente.

In particolare per il primo blocco:
$$c[0] = E_K(m[0] \oplus IV)$$
Per i blocchi successivi:
$$c[i] = E_K(m[i] \oplus c[i-1]), \quad i \geq 1$$
Il punto è lo stesso di quanto visto prima con lo XOR con l'IV: all'inizio otteniamo qualcosa di apparentemente casuale (se l'IV è una buona PRP) criptando $m[0] \oplus IV$, e iterativamente sfruttiamo questo risultato pseudocasuale per randomizzare il blocco successivo.  
La chiave resta la stessa per tutti i blocchi, cambia solo il valore con cui ogni ciphertext viene combinato!

Il messaggio trasmesso è quindi $CT = IV || c[0] || c[1] || ... || c[l-1]$, serve quindi solo l'IV iniziale, ottenendo un overhead di un solo blocco (oltre all'eventuale padding se il messaggio non è multiplo della dimensione del blocco).

A questo punto se due blocchi di plaintext sono uguali ma i rispettivi blocchi di ciphertext che li precede sono diversi, allora i due blocchi di ciphertext risultanti saranno diversi ($c_i = E_K(m[i] \oplus c[i-1])$, se $m[i] = m[j]$ ma $c[i-1] \neq c[j-1]$ allora $c[i] \neq c[j]$).  

Ciò non significa quindi che in CBC è matematicamente impossibile che due blocchi di plaintext uguali producano due blocchi di ciphertext uguali, ma in questo caso la probabilità è trascurabile e, poiché la sicurezza si basa su ciò che un attaccante efficiente può dedurre tenendo conto anche di questo, CBC è considerato sicuro.

##### Decifratura in CBC, requisiti e limiti.
Per il primo blocco:
$$m[0] = D_K(c[0]) \oplus IV$$
Per quelli successivi:
$$m[i] = D_K(c[i]) \oplus c[i-1], \quad i \geq 1$$
infatti $D_K(c[i]) = m[i] \oplus c[i-1]$.

Qui emerge una differenza importante tra cifratura e decifratura. 
- **la cifratura CBC è sequenziale**: per cifrare il blocco $i$ occorre conoscere il ciphertext del blocco precedente $c[i-1]$, quindi non è possibile parallelizzare la cifratura dei blocchi.
- **la decifratura CBC è invece parallelizzabile**: per decifrare il blocco $i$ occorre conoscere solo il ciphertext del blocco precedente $c[i-1]$, quindi se si hanno tutti i ciphertext disponibili è possibile decifrare tutti i blocchi contemporaneamente. 

La cifratura CBC è quindi più lenta della decifratura.

**Requisiti CBC**: 
1. **L'IV iniziale deve essere scelto correttamente**: fondamentale la sua imprevedibilità! Vedremo BEAST contro TLS come attacco che sfrutta la prevedibilità dell'IV iniziale più avanti;
2. **Cifratura non parallelizzabile**
3. **Occorre sia la trasformazione diretta che quella inversa**: $E_K$ per cifrare, $D_K$ per decifrare. In hardware supportare entrambe può richiedere circuteria aggiuntiva. Le modalità successive vedremo usare invece solo $E_K$ per cifrare e decifrare
4. **Occorre completare l'ultimo blocco (padding)**

Approfondiamo l'ultimo requisito. CBC standard richiede blocchi completi per funzionare. Se si usa ad esempio AES (che supporta 16 byte a blocco) e si vuole mandare un msg da 24 byte --> dobbiamo aggiungere 8 byte di padding per completare il blocco finale.

Fondamentale però che il padding inserito **permetta al destinatario di riconoscere e rimuovere il riempimento senza confonderlo con i dati originali**.  
Un esempio è il padding PKCS#7: se mancano $p$ byte per completare il blocco, si aggiungono $p$ byte tutti con valore $p$ (es. $p = 8$ --> aggiungo `08 08 08 08 08 08 08 08`).  
Se il messaggio è già multiplo di 16 byte, PKCS#7 aggiunge un blocco intero di padding da 16 byte, tutti con valore `10`. Questo rende la rimozione non ambigua.

Si anticipano i **padding oracle attacks**: un sistema può rivelare informazioni tramite il modo in cui gestisce ciphertext modificati e errori di padding.

**Esercizio ECB e CBC**: dato il messaggio
$$M = 001 || 001 || 001$$
E le associazioni della permutazione:

<img src="imgs/cbc_ecb.png" width="150">

calcolare il ciphertext in ECB e in CBC, assumendo che l'IV iniziale sia `010`.

**Svolgimento**:
- **ECB**: cifriamo ogni blocco indipendentemente:
    $$CT_{ECB} = E_K(001) || E_K(001) || E_K(001) = 010 || 010 || 010$$
    la ripetizione del plaintext rimane visibile!
- **CBC**: cifriamo il primo blocco con l'IV, poi ogni blocco successivo con il ciphertext precedente:

    <img src="imgs/cbc_ecb_svolgimento.png" width="450">

#### CFB (Cipher Feedback) e OFB (Output Feedback)